# 🎓 India Student Migration & Education Hubs — EDA Notebook

**Purpose:** Interactive exploratory analysis for presentation preparation.

**Run `main.py` first** to generate `data/processed/master_dataset.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
print('✅ Libraries loaded')

## 1. Load Processed Data

In [ ]:
DATA = Path('../data/processed')

master    = pd.read_csv(DATA / 'master_dataset.csv')
nirf      = pd.read_csv(DATA / 'nirf_clean.csv')
aishe     = pd.read_csv(DATA / 'aishe_clean.csv')
migration = pd.read_csv(DATA / 'migration_clean.csv')

print(f'Master:    {master.shape}')
print(f'NIRF:      {nirf.shape}')
print(f'AISHE:     {aishe.shape}')
print(f'Migration: {migration.shape}')

## 2. Dataset Overview

In [ ]:
print('\n=== MASTER DATASET OVERVIEW ===')
master.describe().round(2)

In [ ]:
print('Missing values per column:')
master.isnull().sum()[master.isnull().sum() > 0]

## 3. Migration Analysis

In [ ]:
# Hub vs Exporter split
categories = migration['hub_category'].value_counts()
print('Hub Category Distribution:')
print(categories)

fig, ax = plt.subplots(figsize=(8, 4), facecolor='#0F0F1A')
ax.set_facecolor('#1A1A2E')
colors = ['#6C5CE7','#00B894','#00CEC9','#74B9FF','#FDCB6E','#D63031']
categories.plot(kind='bar', ax=ax, color=colors[:len(categories)], edgecolor='none')
ax.set_title('Hub Category Distribution', color='white', fontsize=13)
ax.tick_params(colors='white')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 hubs vs exporters side-by-side
top_hubs = migration.nlargest(10, 'net_migration_score')[['state','net_migration_score','hub_category']]
top_exp  = migration.nsmallest(10, 'net_migration_score')[['state','net_migration_score','hub_category']]

print('TOP 10 HUBS:')
print(top_hubs.to_string(index=False))
print('\nTOP 10 EXPORTERS:')
print(top_exp.to_string(index=False))

## 4. NIRF Institution Analysis

In [ ]:
# Institutions per state
state_counts = nirf.groupby('state').size().sort_values(ascending=False)
print('Top 15 states by NIRF-ranked institutions:')
print(state_counts.head(15))

In [ ]:
# Category breakdown
cat_counts = nirf.groupby(['category','type']).size().unstack(fill_value=0)
print('\nInstitution count by category × type:')
cat_counts

In [ ]:
# Score distribution by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0F0F1A')

for ax in axes:
    ax.set_facecolor('#1A1A2E')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#636E72')

# Box plot: score by category
categories_list = nirf['category'].unique()
data_by_cat = [nirf[nirf['category']==c]['score'].dropna() for c in categories_list]
bp = axes[0].boxplot(data_by_cat, labels=categories_list, patch_artist=True)
colors_box = ['#6C5CE7','#00CEC9','#FD79A8','#FDCB6E','#00B894']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0].set_title('NIRF Score Distribution by Category', color='white', fontsize=11)
axes[0].set_xticklabels(categories_list, rotation=20, ha='right')

# Govt vs Private score
govt_scores = nirf[nirf['type']=='Government']['score'].dropna()
pvt_scores  = nirf[nirf['type']=='Private']['score'].dropna()
axes[1].hist(govt_scores, bins=15, alpha=0.7, color='#00CEC9', label='Government', edgecolor='none')
axes[1].hist(pvt_scores,  bins=15, alpha=0.7, color='#FD79A8', label='Private',    edgecolor='none')
axes[1].set_title('NIRF Score: Govt vs Private', color='white', fontsize=11)
axes[1].legend(facecolor='#1A1A2E', labelcolor='white')

fig.patch.set_facecolor('#0F0F1A')
plt.tight_layout()
plt.show()

## 5. Enrollment & GER Analysis

In [ ]:
# Top states by GER
ger_top = aishe.nlargest(15, 'gross_enrolment_ratio')[['state','gross_enrolment_ratio','region']]

fig = px.bar(
    ger_top, x='gross_enrolment_ratio', y='state', orientation='h',
    color='region', title='Top 15 States by Gross Enrolment Ratio',
    labels={'gross_enrolment_ratio': 'GER (%)', 'state': ''},
    template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(height=450, showlegend=True)
fig.show()

## 6. Correlation Deep Dive

In [ ]:
# What drives net migration score?
numeric = master.select_dtypes(include='number').drop(
    columns=['net_migration_norm','ger_norm','nirf_norm','priv_norm'], errors='ignore'
)

if 'net_migration_score' in numeric.columns:
    corr_with_migration = numeric.corr()['net_migration_score'].drop('net_migration_score')
    corr_with_migration = corr_with_migration.sort_values(key=abs, ascending=False)
    
    print('Pearson correlations with net_migration_score:')
    for feat, val in corr_with_migration.items():
        bar = '█' * int(abs(val) * 25)
        sign = '+' if val > 0 else '-'
        print(f'  {feat:<35} {sign}{abs(val):.3f}  {bar}')

## 7. Education Hub Index — Final Ranking

In [ ]:
hub_ranking = (
    master[['state','education_hub_index','hub_category','region',
            'gross_enrolment_ratio','net_migration_score','total_enrollment_k']]
    .dropna(subset=['education_hub_index'])
    .sort_values('education_hub_index', ascending=False)
    .reset_index(drop=True)
)
hub_ranking.index = hub_ranking.index + 1
hub_ranking.head(15)

In [ ]:
# Interactive bubble: Hub Index vs GER
fig = px.scatter(
    hub_ranking.head(25),
    x='gross_enrolment_ratio',
    y='net_migration_score',
    size='total_enrollment_k',
    color='region',
    text='state',
    title='Education Hub Landscape — Top 25 States',
    template='plotly_dark',
    size_max=55,
)
fig.update_traces(textposition='top center')
fig.add_hline(y=0, line_dash='dash', opacity=0.5)
fig.update_layout(height=550)
fig.show()

## 8. Conclusions

Run `main.py` for full written conclusions in `presentation/insights_report.md`.

Quick summary from EDA:
- **Delhi + South India = dominant hubs**
- **Bihar + North-East = structural exporters**  
- **GER is strongest predictor** of hub status
- **Govt IITs/NITs** carry disproportionate prestige signal
- **Tamil Nadu private model** = largest experiment in mass-scale higher education